# Temă de Programare: Auditul Valorilor Lipsă

Bun venit la tema **Auditul Valorilor Lipsa**!

Datele lipsa reprezinta una dintre cele mai raspandite provocari din machine learning-ul din lumea reala. Inainte sa poti construi orice model, trebuie sa intelegi **amploarea** si **tiparele** valorilor lipsa din datele tale. Acest notebook te invata cum sa faci, in mod sistematic, un audit al valorilor lipsa dintr-un set de date.

Asa cum a fost prezentat in teorie, valorile lipsa din seturile de date reale nu apar doar ca `NaN`. Placeholder-ele numerice precum `-1`, `999` sau `-9999` sunt frecvente in sistemele legacy; placeholder-ele text precum `"unknown"` sau `"null"` apar in coloanele text; sirurile goale pot trece neobservate prin parser-ele CSV. Un pas complet de detectie combina `df.isna()` cu verificari dependente de domeniu pentru valori in afara intervalului sau semantic goale.

**Ce vei face in aceasta tema:**

* Vei numara valorile lipsa pe coloana folosind `.isnull().sum()`
* Vei calcula procentele valorilor lipsa raportat la dimensiunea setului de date
* Vei identifica tipare de co-aparitie: ce randuri au mai multe coloane lipsa simultan
* Vei crea un raport de audit cuprinzator pentru valorile lipsa
* Vei detecta coloanele care depasesc un prag ridicat de lipsa

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt blocate cu excepția celor în care trebuie să trimiți soluțiile sau atunci când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga sau modifica niciun cod care se află în afara acestor comentarii**.

* Poți adăuga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, deci nu te baza pe celule create nou pentru a găzdui codul soluției — folosește locurile prevăzute în acest scop.

---

## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea Valorilor Lipsa](#1---understanding-missing-values)
    - [1.1 - Tipuri de Lipsa (MCAR, MAR, MNAR)](#11---types-of-missingness)
    - [1.2 - Set de date exemplu](#12---sample-data)
- [2 - Numararea Valorilor Lipsa](#2---counting-missing-values)
    - **[Exercitiul 1 - `count_missing_values`](#exercise-1---count_missing_values)**
- [3 - Calcularea Procentelor de Lipsa](#3---calculating-missing-percentages)
    - **[Exercitiul 2 - `calculate_missing_percentage`](#exercise-2---calculate_missing_percentage)**
- [4 - Identificarea Tiparelor de Lipsa](#4---identifying-missing-patterns)
    - **[Exercitiul 3 - `identify_missing_patterns`](#exercise-3---identify_missing_patterns)**
- [5 - Crearea unui Rezumat al Valorilor Lipsa](#5---creating-a-missing-value-summary)
    - **[Exercitiul 4 - `create_missing_summary`](#exercise-4---create_missing_summary)**
- [6 - Detectarea Lipsei Ridicate](#6---detecting-high-missingness)
    - **[Exercitiul 5 - `detect_high_missingness`](#exercise-5---detect_high_missingness)**
- [7 - Vizualizarea Datelor Lipsa](#7---visualizing-missing-data)


<a name='imports'></a>
## Importuri

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-missing-values'></a>
## 1 - Înțelegerea Valorilor Lipsă

<a name='11---types-of-missingness'></a>
### 1.1 - Tipuri de Lipsa (MCAR, MAR, MNAR)

Inainte sa intram in cod, este esential sa intelegem ca valorile lipsa apar prin mecanisme diferite, fiecare avand implicatii diferite asupra modului in care trebuie tratate:

1. **MCAR (Missing Completely At Random)**: Probabilitatea ca o valoare sa lipseasca este constanta si independenta de toate celelalte variabile, atat observate, cat si neobservate. Formal: $P(R=0 | X_{obs}, X_{mis}) = P(R=0)$. Exemplu: un senzor nu reuseste aleator sa inregistreze o masuratoare.

2. **MAR (Missing At Random)**: Probabilitatea lipsei depinde de *alte variabile observate*, dar nu de valoarea lipsa in sine. Formal: $P(R=0 | X_{obs}, X_{mis}) = P(R=0 | X_{obs})$. Exemplu: respondentii mai tineri au o probabilitate mai mica sa raporteze venitul, dar dat fiind varsta, probabilitatea ca venitul sa lipseasca nu depinde de valoarea reala a venitului.

3. **MNAR (Missing Not At Random)**: Probabilitatea lipsei depinde chiar de valoarea lipsa. Formal: $P(R=0 | X_{obs}, X_{mis}) = P(R=0 | X_{mis})$. Exemplu: persoanele cu venit mare sunt mai putin dispuse sa isi declare venitul *pentru ca* acesta este mare.

Intelegerea mecanismului este primul pas in alegerea strategiei corecte de tratare. MCAR este singurul caz in care eliminarea randurilor incomplete nu introduce bias. MAR permite imputare bazata pe model. MNAR este cel mai periculos si necesita expertiza de domeniu.


<a name='12---sample-data'></a>
### 1.2 - Set de Date Exemplu

In aceasta tema vom folosi un set de date sintetic despre clienti, generat cu tipare cunoscute de lipsa. Setul de date reproduce tipurile de tipare discutate in teorie:
- lipsa **MCAR** in `age` (10% aleator)
- lipsa **MAR** in `income` (mai ridicata pentru someri/studenti)
- lipsa **structurala** in `years_experience` (studentii nu pot avea experienta de munca)
- lipsa **aleatoare** in `satisfaction_score` (15%)


In [ ]:
# Generate sample data with missing values
df = helper_utils.generate_sample_data_with_missing(n_rows=100, random_state=42)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
# Get basic info — notice 'Non-Null Count' is less than total rows for some columns
print("Dataset Info:")
df.info()

In [ ]:
# In pandas, NaN is the canonical missing-value token for numeric columns
sample_series = pd.Series([1, 2, np.nan, 4, np.nan])
print(f"Sample series:  {sample_series.values}")
print(f"Is null:        {sample_series.isnull().values}")
print(f"Count of nulls: {sample_series.isnull().sum()}")

<a name='2---counting-missing-values'></a>
## 2 - Numararea Valorilor Lipsa

Primul pas in orice audit al valorilor lipsa este sa numeri cate valori lipsesc in fiecare coloana. `.isnull()` returneaza un DataFrame boolean de aceeasi forma, unde `True` marcheaza fiecare celula lipsa. Aplicarea lui `.sum()` apoi reduce fiecare coloana la un numar total.


In [ ]:
# Example: Counting missing values
example_df = pd.DataFrame({
    'A': [1, 2, np.nan, 4],
    'B': [np.nan, np.nan, 3, 4],
    'C': [1, 2, 3, 4]
})
print("Example DataFrame:")
print(example_df)
print("\nMissing values per column:")
print(example_df.isnull().sum())

<a name='exercise-1---count_missing_values'></a>
### **Exercitiul 1 - `count_missing_values`**

**Sarcina ta:**

Implementeaza `count_missing_values` astfel incat sa returneze un `pd.Series` unde indexul este format din numele coloanelor, iar valorile reprezinta numarul de valori lipsa.

**Cerinte:**
- Returneaza un `pd.Series` (nu un `pd.DataFrame`)
- Indexul trebuie sa fie format din numele coloanelor DataFrame-ului de intrare
- Valorile trebuie sa fie intregi nenegativi

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Foloseste `df.isnull()` pentru a crea o masca booleana unde `True` = lipsa

**Pasul 2:** Lantuieste `.sum()` pentru a numara valorile `True` pe fiecare coloana

**Formula:** `missing_counts = df.isnull().sum()`

</details>


In [ ]:
# GRADED FUNCTION: count_missing_values
def count_missing_values(df):
    """
    Count the number of missing values in each column of a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.Series: Series with column names as index and missing counts as values.
    """
    ### ÎNCEPUT DE COD AICI ###

    missing_counts = None

    ### SFÂRȘIT DE COD AICI ###

    return missing_counts

In [ ]:
# Try your implementation
missing_counts = count_missing_values(df)
print("Missing value counts per column:")
print(missing_counts)
print(f"\nTotal missing values: {missing_counts.sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_1(count_missing_values)

<a name='3---calculating-missing-percentages'></a>
## 3 - Calcularea Procentelor de Lipsa

Numaratorile brute sunt utile, dar procentele iti ofera o imagine mai buna asupra *severitatii* raportat la dimensiunea setului de date. Conform pragurilor larg utilizate din teorie:

| Missing % | Abordare tipica |
|-----------|-----------------|
| < 5%      | Sigur de imputat cu metode simple sau se pot elimina randuri |
| 5% – 20%  | Necesita alegere atenta a metodei; ia in calcul analiza tiparelor |
| 20% – 50% | Se recomanda imputare avansata sau variabile indicator |
| > 50%     | Ia in calcul eliminarea coloanei; riscul imputarii este ridicat |


In [ ]:
# Example: Converting counts to percentages
example_counts = pd.Series({'A': 25, 'B': 50, 'C': 0}, name='missing_count')
total_rows = 100
percentages = (example_counts / total_rows) * 100
print(f"Missing counts:  {example_counts.values}")
print(f"As percentages:  {percentages.values}")

<a name='exercise-2---calculate_missing_percentage'></a>
### **Exercitiul 2 - `calculate_missing_percentage`**

**Sarcina ta:**

Implementeaza `calculate_missing_percentage` astfel incat sa calculeze procentul de valori lipsa pe fiecare coloana.

**Cerinte:**
- Returneaza un `pd.Series` cu procente pe scala **0–100** (nu 0–1)
- Valorile trebuie sa reprezinte ce fractie din randurile acelei coloane este lipsa

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Numara valorile lipsa folosind `df.isnull().sum()`

**Pasul 2:** Imparte la `len(df)` pentru a obtine fractia

**Pasul 3:** Inmulteste cu `100` pentru a converti la procent

**Formula:** `percentage = (df.isnull().sum() / len(df)) * 100`

</details>


In [ ]:
# GRADED FUNCTION: calculate_missing_percentage
def calculate_missing_percentage(df):
    """
    Calculate the percentage of missing values in each column.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.Series: Series with column names as index and missing percentages (0-100) as values.
    """
    ### ÎNCEPUT DE COD AICI ###

    missing_percentage = None

    ### SFÂRȘIT DE COD AICI ###

    return missing_percentage

In [ ]:
missing_pct = calculate_missing_percentage(df)
print("Missing value percentages per column:")
print(missing_pct.round(2))

In [ ]:
# Testează-ți codul!
unittests.exercise_2(calculate_missing_percentage)

<a name='4---identifying-missing-patterns'></a>
## 4 - Identificarea Tiparelor de Lipsa

Dincolo de simpla numarare, este util sa intelegi **tiparele de co-aparitie**: ce randuri au mai multe coloane lipsa in acelasi timp? Un rand in care lipsesc simultan 5 coloane sugereaza o eroare de ingestie a datelor, nu zgomot aleator.

Graficul `missingno` de tip matrice din teorie afiseaza **benzi orizontale albe** atunci cand randuri intregi pierd mai multe valori, un semn clasic al unei defectiuni la nivel de batch.


In [ ]:
# Example: Number of missing values per row
example_df = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan],
    'B': [np.nan, np.nan, 3, 4],
    'C': [1, 2, 3, 4]
})
print("DataFrame:")
print(example_df)
print("\nMissing values per row:")
print(example_df.isnull().sum(axis=1))

<a name='exercise-3---identify_missing_patterns'></a>
### **Exercitiul 3 - `identify_missing_patterns`**

**Sarcina ta:**

Implementeaza `identify_missing_patterns` astfel incat sa returneze un DataFrame care descrie *ce* coloane lipsesc in fiecare rand care contine cel putin o valoare lipsa.

**Cerinte:**
- Returneaza un `pd.DataFrame` doar cu acele randuri in care lipseste cel putin o valoare
- Include o coloana `'missing_columns'` care contine lista numelor de coloane ce sunt `NaN` pe acel rand
- Include o coloana `'missing_count'` cu numarul intreg de coloane lipsa pe acel rand

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Gaseste randurile cu cel putin o valoare lipsa: `df.isnull().any(axis=1)` (serie booleana)

**Pasul 2:** Obtine indicii acestor randuri

**Pasul 3:** Pentru fiecare index de rand, identifica ce coloane sunt nule:
```python
missing_cols = df.columns[df.loc[idx].isnull()].tolist()
```

**Pasul 4:** Construieste o lista de dictionare, apoi `pd.DataFrame(patterns).set_index('row_index')`

</details>


In [ ]:
# GRADED FUNCTION: identify_missing_patterns
def identify_missing_patterns(df):
    """
    Identify rows with missing values and describe their missingness pattern.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: DataFrame indexed by original row index, with columns:
            - 'missing_columns' (list): column names that are missing in that row
            - 'missing_count' (int): number of missing values in that row
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Boolean mask — True for rows that have at least one missing value
    rows_with_missing = None

    # Step 2: Get original row indices
    missing_indices = None

    # Step 3: Build pattern list
    patterns = []
    for idx in missing_indices:
        row = df.loc[idx]
        missing_cols = None  # list of column names that are null
        missing_count = None  # integer count
        patterns.append({
            'row_index': idx,
            'missing_columns': missing_cols,
            'missing_count': missing_count
        })

    # Step 4: Create result DataFrame
    result = None

    ### SFÂRȘIT DE COD AICI ###

    return result

In [ ]:
patterns = identify_missing_patterns(df)
print(f"Rows with at least one missing value: {len(patterns)}")
print("\nFirst 10 rows with missing patterns:")
patterns.head(10)

In [ ]:
# Testează-ți codul!
unittests.exercise_3(identify_missing_patterns)

<a name='5---creating-a-missing-value-summary'></a>
## 5 - Crearea unui Rezumat al Valorilor Lipsa

Pentru un audit profesionist al valorilor lipsa, combini toate metricele intr-un singur tabel, analog functiei `missing_audit()` prezentate in teorie. Astfel poti prioritiza rapid coloanele care necesita atentie si vezi si contextul tipului de date.


In [ ]:
# Example of the target output format (from theory)
demo = pd.DataFrame({'missing_count': [4, 2, 1, 0],
                     'missing_%':     [40.0, 20.0, 10.0, 0.0],
                     'dtype':         ['float64', 'float64', 'float64', 'int64']},
                    index=['credit_score', 'income', 'age', 'customer_id'])
print(demo.to_string())

<a name='exercise-4---create_missing_summary'></a>
### **Exercitiul 4 - `create_missing_summary`**

**Sarcina ta:**

Implementeaza `create_missing_summary` astfel incat sa creeze un raport cuprinzator pe coloane, ordonat dupa severitatea lipsei.

**Cerinte:**
- Returneaza un `pd.DataFrame` cu un rand pentru fiecare coloana
- Include coloanele: `'missing_count'`, `'missing_percentage'`, `'dtype'`
- Sorteaza descrescator dupa `'missing_count'`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Calculeaza numarul de valori lipsa: `df.isnull().sum()`

**Pasul 2:** Calculeaza procentele valorilor lipsa: `(df.isnull().sum() / len(df)) * 100`

**Pasul 3:** Obtine tipurile de date: `df.dtypes`

**Pasul 4:** Combina intr-un DataFrame si sorteaza:
```python
summary = pd.DataFrame({'missing_count': ..., 'missing_percentage': ..., 'dtype': ...})
summary.sort_values('missing_count', ascending=False)
```

</details>


In [ ]:
# GRADED FUNCTION: create_missing_summary
def create_missing_summary(df):
    """
    Create a comprehensive missing value summary for all columns.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: DataFrame with columns 'missing_count', 'missing_percentage', 'dtype',
                      sorted by missing_count descending.
    """
    ### ÎNCEPUT DE COD AICI ###

    missing_count = None
    missing_percentage = None
    dtype = None

    summary = None

    ### SFÂRȘIT DE COD AICI ###

    return summary

In [ ]:
summary = create_missing_summary(df)
print("Missing Value Summary:")
summary

In [ ]:
# Testează-ți codul!
unittests.exercise_4(create_missing_summary)

<a name='6---detecting-high-missingness'></a>
## 6 - Detectarea Lipsei Ridicate

Teoria propune ca acele coloane cu **> 50%** valori lipsa sunt candidate pentru eliminare, mai degraba decat pentru imputare: completarea a mai mult de jumatate dintr-o coloana cu valori inferate introduce un risc substantial. Totusi, contextul conteaza: o rata de 55% intr-un set cu un milion de randuri este mai putin alarmanta decat aceeasi rata intr-un set cu 500 de randuri.


In [ ]:
# Example: Filter columns by missingness threshold
example_pct = pd.Series({'income': 22.0, 'age': 5.0, 'score': 55.0, 'id': 0.0})
threshold = 20.0
high_missing = example_pct[example_pct > threshold]
print(f"Columns above {threshold}% missing threshold:")
print(high_missing)

<a name='exercise-5---detect_high_missingness'></a>
### **Exercitiul 5 - `detect_high_missingness`**

**Sarcina ta:**

Implementeaza `detect_high_missingness` astfel incat sa returneze o lista cu numele coloanelor al caror procent de lipsa depaseste un prag dat.

**Cerinte:**
- Accepta un parametru `threshold` (float, implicit `20.0`)
- Returneaza o **lista** cu nume de coloane (siruri de caractere) care depasesc pragul
- Lista poate fi vida daca nicio coloana nu depaseste pragul

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Calculeaza procentul de lipsa pe fiecare coloana

**Pasul 2:** Filtreaza: `missing_pct[missing_pct > threshold]`

**Pasul 3:** Returneaza `.index.tolist()`

</details>


In [ ]:
# GRADED FUNCTION: detect_high_missingness
def detect_high_missingness(df, threshold=20.0):
    """
    Detect columns whose missing percentage exceeds a given threshold.

    Args:
        df (pd.DataFrame): Input DataFrame.
        threshold (float): Percentage threshold (0-100). Default is 20.0.

    Returns:
        list: Column names with missing percentage greater than the threshold.
    """
    ### ÎNCEPUT DE COD AICI ###

    missing_pct = None
    high_missing_cols = None

    ### SFÂRȘIT DE COD AICI ###

    return high_missing_cols

In [ ]:
high_missing = detect_high_missingness(df, threshold=10.0)
print(f"Columns with > 10% missing: {high_missing}")

high_missing_20 = detect_high_missingness(df, threshold=20.0)
print(f"Columns with > 20% missing: {high_missing_20}")

In [ ]:
# Testează-ți codul!
unittests.exercise_5(detect_high_missingness)

<a name='7---visualizing-missing-data'></a>
## 7 - Vizualizarea Datelor Lipsa

Teoria acopera patru vizualizari `missingno`:
- **Bar plot** — numarul de valori nenule pe coloana, dintr-o privire
- **Matrix plot** — grila binara care arata exact unde lipsesc valorile; sparkline-ul din dreapta rezuma completitudinea pe rand
- **Heatmap** — corelatia perechilor de tipare de lipsa (−1 la +1); coloanele care lipsesc mereu impreuna → +1, cele care nu lipsesc niciodata impreuna → −1
- **Dendrogram** — clusterizare ierarhica a coloanelor dupa similaritatea tiparelor de lipsa

Ruleaza celulele de mai jos pentru a explora aceste vizualizari pe setul de date exemplu.


In [ ]:
# Seaborn / matplotlib missingness heatmap
helper_utils.plot_missing_heatmap(df)

In [ ]:
# Bar chart — missing count per column
helper_utils.plot_missing_bar(df, show_percentage=True)

In [ ]:
# Matrix plot — binary presence/absence grid
helper_utils.plot_missing_matrix(df)

In [ ]:
# Pairwise missingness correlation
helper_utils.plot_missing_correlation(df)

In [ ]:
# Full textual summary of missing patterns
helper_utils.summarize_missing_patterns(df)